In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "abc123"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.criteria({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)

#df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash("main")

image_filename = 'Grazioso Salvare Logo.png' # puts in the Grazioso logo
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Joshua Ladue CS-340 Dashboard'))),#My name at the top center
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())),#Grazioso image under
    html.Hr(),
    html.Div(
        

className='row',
        style={'display': 'flex'},
        children=[
            dcc.RadioItems(#radio buttons to filter the animals into types
                id='animal-filter',
                options=[
                    {'label': 'All Animals', 'value': 'ALL'},
                    {'label': 'Dogs', 'value': 'Dog'},
                    {'label': 'Cats', 'value': 'Cat'}
    ],
    value='ALL',
    labelStyle={'display': 'inline-block', 'margin-right': '20px'}
)
                    
        ]       
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),

    #dff = pd.DataFrame.from_dict(viewData),
                         
    editable=True,
        row_selectable="single",
        selected_rows=[],
        filter_action="native",#Native filtering
        sort_action="native", #Native sorting
        page_action="native",#Native pages
        page_current=0,
        page_size=5,
# Because we only allow single row selection, the list can 
# be converted to a row index here

# Austin TX is at [30.75,-97.48]
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    Input('animal-filter', 'value')
)
def update_dashboard(selected_value):#filtering options for the dashboard
    if selected_value == 'Dog':#filters to only dogs
        df = pd.DataFrame.from_records(
            db.criteria({'animal_type': 'Dog'})
        )

    elif selected_value == 'Cat':#filters to cats
        df = pd.DataFrame.from_records(
            db.criteria({'animal_type': 'Cat'})
        )

    else:
        df = pd.DataFrame.from_records(#else it will be all animals
            db.criteria({})
        )

    return df.to_dict('records')#returns the results
#
#        

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    df = pd.DataFrame.from_dict(viewData)

    fig = px.pie(#the pie graph of breeds available
        df,
        names='breed',
        title='Percentage of Breeds Available'
    )

    return [dcc.Graph(figure=fig)]#returns the graph
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:#Helps stop any error for being 0
        return []

    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:#returns nothing if there is no data
        return []

    if index is None or len(index) == 0:#if there is no index at the start it will start at the first one
        row = 0
    else:
        row = index[0]

    dff = pd.DataFrame.from_dict(viewData)

    # Austin TX is at [30.75,-97.48]
    return [#Code for the map
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[#style variables for the map
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() #runs the server

Dash app running on https://salsasocial-gyronelson-3000.codio.io/proxy/8050/
